# Aligning MERFISH to the Allen Brain Atlas (squidpy + STalign)

In this notebook, we align a MERFISH dataset of an adult mouse coronal brain section to the Allen Brain Atlas.

**Note:** 3D atlas-to-2D-slice alignment (`LDDMM_3D_to_slice`) is not yet available in the squidpy high-level API. This notebook uses the STalign functions directly from squidpy's internal LDDMM module. For standard 2D point-to-point alignments, see the moscot-based notebooks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nrrd
import torch
import squidpy as sq

# Use STalign internals for 3D alignment (not yet in sq.experimental.tl.align)
from squidpy.experimental._lddmm import rasterize
from squidpy.experimental._lddmm._transforms import L_T_from_points

# Also need the original STalign for LDDMM_3D_to_slice
# which is not yet ported to squidpy
from STalign import STalign

## Load MERFISH data

In [ ]:
df = pd.read_csv('../merfish_data/s1r1_metadata.csv.gz')
x = np.array(df['center_x'])
y = np.array(df['center_y'])

dx = 10
blur = 1

## Load Allen Brain Atlas

This section remains identical to the STalign version as the 3D atlas alignment is not yet in the squidpy API.

In [ ]:
url = 'http://api.brain-map.org/api/v2/data/query.csv?criteria=model::Structure,rma::criteria,[ontology_id$eq1],rma::options[order$eq%27structures.graph_order%27][num_rows$eqall]'
ontology_name, namesdict = STalign.download_aba_ontology(url, 'allen_ontology.csv')

In [ ]:
imageurl = 'http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/ara_nissl/ara_nissl_50.nrrd'
labelurl = 'http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/annotation_50.nrrd'
imagefile, labelfile = STalign.download_aba_image_labels(imageurl, labelurl, 'aba_nissl.nrrd', 'aba_annotation.nrrd')

In [ ]:
# Rasterize using squidpy's rasterize function
X_, Y_, W = rasterize(x, y, dx=dx, blur=blur)

## Initialize alignment parameters

The 3D atlas alignment procedure uses `LDDMM_3D_to_slice` which is specific to STalign.

In [ ]:
slice_idx = 177

vol, hdr = nrrd.read(imagefile)
A = vol
vol, hdr = nrrd.read(labelfile)
L = vol

dxA = np.diag(hdr['space directions'])
nxA = A.shape
xA = [np.arange(n)*d - (n-1)*d/2.0 for n, d in zip(nxA, dxA)]

In [ ]:
# Alignment parameters
points_atlas = np.array([[0, 2580]])
points_target = np.array([[8, 2533]])
Li, Ti = STalign.L_T_from_points(points_atlas, points_target)

xJ = [Y_, X_]
J = W[None] / np.mean(np.abs(W))
xI = xA
I = A[None] / np.mean(np.abs(A), keepdims=True)
I = np.concatenate((I, (I - np.mean(I))**2))

In [ ]:
sigmaA = 2
sigmaB = 2
sigmaM = 2
muA = torch.tensor([3, 3, 3], device='cpu')
muB = torch.tensor([0, 0, 0], device='cpu')

scale_x = 0.9
scale_y = 0.9
scale_z = 0.9
theta0 = 0

T = np.array([-xI[0][slice_idx], np.mean(xJ[0])-(Ti[0]*scale_y), np.mean(xJ[1])-(Ti[1]*scale_x)])
scale_atlas = np.array([[scale_z, 0, 0], [0, scale_x, 0], [0, 0, scale_y]])
L_mat = np.array([[1.0, 0.0, 0.0], [0.0, np.cos(theta0), -np.sin(theta0)], [0.0, np.sin(theta0), np.cos(theta0)]])
L_mat = np.matmul(L_mat, scale_atlas)

## Run 3D-to-slice alignment

This uses the STalign `LDDMM_3D_to_slice` function directly.

In [ ]:
%%time
transform = STalign.LDDMM_3D_to_slice(
    xI, I, xJ, J,
    T=T, L=L_mat,
    nt=4, niter=2000,
    device='cpu',
    sigmaA=sigmaA, sigmaB=sigmaB, sigmaM=sigmaM,
    muA=muA, muB=muB,
)

In [ ]:
A_out = transform['A']
v = transform['v']
xv = transform['xv']
Xs = transform['Xs']

## Analyze results

In [ ]:
df_result = STalign.analyze3Dalign(
    labelfile, xv, v, A_out, xJ, dx,
    scale_x=scale_x, scale_y=scale_y,
    x=x, y=y, X_=X_, Y_=Y_,
    namesdict=namesdict, device='cpu',
)
df_result

In [ ]:
STalign.plot_brain_regions(df_result)